In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [ ]:
import sys
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from spice import SpiceEstimator, SpiceConfig, csv_to_dataset, BaseModel, plot_session, split_data_along_sessiondim, SpiceDataset

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training

# Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [ ]:
path_data = '../data/ganesh2024a/ganesh2024a_choice.csv'

# Load your data
dataset = csv_to_dataset(
    file = path_data,
    df_participant_id='subjID',
    df_choice='choice',
    df_feedback='reward',
    df_block='blocks',
    additional_inputs=['contrast_difference'],
    # timeshift_additional_inputs=[-1],
    )

# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (participants*blocks*experiments, trials, timesteps_in_trial, features)
# features are (action_0, action_1, reward_0, reward_1, *additional_inputs, timestep, trial, block, experiment, participant)
 
# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

# add the next contrast difference as the cue for the next choice
# keep the current contrast difference to learn the chosen contrast's reward values
contr_diff = dataset.xs[..., n_actions*2].unsqueeze(-1)
contr_diff_next = contr_diff[:, 1:]
xs = torch.concat((
    dataset.xs[:, :-1, :, :n_actions*2],
    contr_diff[:, :-1, :],
    contr_diff_next,
    dataset.xs[:, :-1, :, 2*n_actions+1:],
), dim=-1)
dataset = SpiceDataset(xs, dataset.ys[:, :-1])

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-2*n_actions-5}")

Now we are going to split the data by blocks with ```split_data_along_sessiondim()``` using 75% as training data and 25% as testing data

In [ ]:
list_test_sessions = (3, 6, 9)
dataset_train, dataset_test = split_data_along_sessiondim(dataset=dataset, list_test_sessions=list_test_sessions)

print(f"Number of sessions in training data: {len(dataset_train)}")
print(f"Number of sessions in test data: {len(dataset_test)}")

# SPICE Setup

Now we are going to define the configuration for SPICE with a `SpiceConfig` object.

The `SpiceConfig` takes as arguments 
1. `library_setup (dict)`: Defining the variable names of each module.
2. `memory_state (dict)`: Defining the memory state variables and their initial values.
3. `states_in_logit (list)`: Defining which of the memory state variables are used later for the logit computation. This is necessary for some background processes.  

And now we are going to define the SPICE model which is a child of the `BaseModel` and `torch.nn.Module` class and takes as required arguments:
1. `spice_config (SpiceConfig)`: previously defined SpiceConfig object
2. `n_actions (int)`: number of possible actions in your dataset (including non-displayed ones if applicable).
3. `n_participants (int)`: number of participants in your dataset.

As usual for a `torch.nn.Module` we have to define at least the `__init__` method and the `forward` method.
The `forward` method gets called when computing a forward pass through the model and takes as inputs `(inputs (SpiceDataset.xs), prev_state (dict, default: None), batch_first (bool, default: False))` and returns `(logits (torch.Tensor, shape: (n_participants*n_blocks*n_experiments, timesteps, n_actions)), updated_state (dict))`. Two necessary method calls inside the forward pass are:
1. `self.init_forward_pass(inputs, prev_state) -> SpiceSignals`: returns a `SpiceSignals` object which carries all relevant information already processed.
2. `self.post_forward_pass(SpiceSignals, batch_first) -> SpiceSignals`: does some re-arranging of the logits to adhere to `batch_first`.

In [ ]:
spice_config = SpiceConfig(
    library_setup={
        'perception_certainty': ['contr_diff[t]'],
        'reward_learning_chosen': ['reward[t]', 'certainty[t]'],
        'reward_learning_unchosen': ['certainty[t]'],
        'choice_persistance': ['choice[t]', 'certainty[t]', 'certainty_next[t+1]'],
    },
    
    memory_state={
        'value_reward_contrast': 0.,
        'value_choice_contrast': 0.,
    },
)

In [ ]:
class SPICERNN(BaseModel):
    
    def __init__(self, deterministic_perception=False, **kwargs):
        super().__init__(**kwargs)
        
        self.dropout = 0.1
        self.deterministic_perception = deterministic_perception
        
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants, embedding_size=self.embedding_size, dropout=self.dropout)
        
        # perception: signed contr_diff → sigmoid
        self.setup_module(key_module='perception_certainty', input_size=1+self.embedding_size, include_state=False, dropout=self.dropout)
        # chosen item: learns from reward, modulated by certainty
        self.setup_module(key_module='reward_learning_chosen', input_size=2+self.embedding_size, dropout=self.dropout)
        # unchosen item: forgetting/persistence dynamics, modulated by certainty
        self.setup_module(key_module='reward_learning_unchosen', input_size=1+self.embedding_size, dropout=self.dropout)
        # choice persistance module
        self.setup_module(key_module='choice_persistance', input_size=3+self.embedding_size, dropout=self.dropout)
        
    def forward(self, inputs, prev_state:torch.Tensor=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        
        # feature extraction
        # scalar contrast difference per trial (T, W, E, B)
        cd_current = spice_signals.additional_inputs[..., 0]
        cd_next = spice_signals.additional_inputs[..., 1].unsqueeze(-1)
        # repeated to n_actions for module inputs (T, W, E, B, n_actions)
        contr_diff_current = cd_current.unsqueeze(-1).repeat(1, 1, 1, 1, self.n_actions)
        contr_diff_next = cd_next.repeat(1, 1, 1, 1, self.n_actions)
        
        # Map actions: position space (left=0, right=1) → item space (low=0, high=1)
        # cd <= 0: left=low, right=high;  cd > 0: left=high, right=low
        chose = spice_signals.actions.argmax(dim=-1)  # (T, W, E, B)
        action_contrast = torch.zeros_like(spice_signals.actions)
        # chose_low: chose left when left=low (cd<=0), OR chose right when right=low (cd>0)
        action_contrast[..., 0] = (((cd_current <= 0) & (chose == 0)) | ((cd_current > 0) & (chose == 1))).float()
        # chose_high: chose right when right=high (cd<=0), OR chose left when left=high (cd>0)
        action_contrast[..., 1] = (((cd_current <= 0) & (chose == 1)) | ((cd_current > 0) & (chose == 0))).float()
        
        # Scalar reward per trial (sum over one-hot reward vector)
        reward_scalar = spice_signals.rewards.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)
        
        participant_embeddings = self.participant_embedding(spice_signals.participant_ids)
        
        for trial in spice_signals.trials:
            
            # --- Perception: p(left=low) for current trial ---
            certainty_current = self.call_module(
                key_module='perception_certainty',
                inputs=contr_diff_current[trial],#.abs(),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
                activation_rnn=torch.nn.functional.sigmoid,
                # activation_rnn=torch.nn.functional.tanh,
            )
            
            certainty_next = self.call_module(
                key_module='perception_certainty',
                inputs=contr_diff_next[trial],#.abs(),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
                activation_rnn=torch.nn.functional.sigmoid,
                # activation_rnn=torch.nn.functional.tanh,
            )

            # --- Reward learning with explicit chosen/unchosen split ---
            # Hard masks use the deterministic item space action (ground truth mapping).
            # Certainty is passed as input so the GRU can learn to attenuate updates
            # when the deterministic assignment is unreliable.
            
            # Chosen item: update value based on actual reward
            self.call_module(
                key_module='reward_learning_chosen',
                key_state='value_reward_contrast',
                action_mask=action_contrast[trial],
                inputs=(
                    reward_scalar[trial], 
                    certainty_current,
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # Unchosen item: can learn forgetting, persistence, or drift
            self.call_module(
                key_module='reward_learning_unchosen',
                key_state='value_reward_contrast',
                action_mask=1 - action_contrast[trial],
                inputs=(certainty_current,),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # Choice persistance
            self.call_module(
                key_module='choice_persistance',
                key_state='value_choice_contrast',
                inputs=(
                    action_contrast[trial],
                    certainty_current,
                    certainty_next,
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # --- Decision: soft-map item values to action space ---
                        
            logits_contrast = self.state['value_reward_contrast'] + self.state['value_choice_contrast']
            
            # deriving p(left=low) from contrast-difference-based certainty to assign learned reward values to given options; range=(0.5, 1.0)
            # TODO: add sign information about contrast difference
            # certainty_next = (certainty_next + 1) / 2
            certainty_next = certainty_next / 2 + 0.5
            mixed_logits_item_space = logits_contrast * certainty_next + logits_contrast.flip(-1) * (1 - certainty_next)
            
            # map mixed logits from item space (low, high) into action space (left, right)
            spice_signals.logits[trial] = torch.where(cd_next[trial] < 0, mixed_logits_item_space, mixed_logits_item_space.flip(-1))
            
        spice_signals = self.post_forward_pass(spice_signals)
        
        return spice_signals.logits, self.get_state()

Let's setup now the `SpiceEstimator` object and fit it to the data!

We are going to fit SPICE once without SINDy as a regularizer for the RNN (`sindy_weight=0`) to get peak SPICE performance given the architecture.

Then we are fitting SPICE in the normal way to get the interpretable model.

In [ ]:
estimator = SpiceEstimator(
    # model paramaeters
    spice_class=SPICERNN,
    spice_config=spice_config,
    n_actions=n_actions,
    n_participants=n_participants,
    
    epochs=500,
    sindy_weight=0,
    sindy_library_polynomial_degree=2,
    
    compiled_forward=False,
    verbose=True,
)

print(f"\nStarting training on {estimator.device}...")
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)
print("\nTraining complete!")

In [ ]:
path_spice = '../params/ganesh2024a/spice_ganesh2024a.pkl'

estimator = SpiceEstimator(
    # model paramaeters
    spice_class=SPICERNN,
    spice_config=spice_config,
    n_actions=n_actions,
    n_participants=n_participants,
    
    epochs=1000,
    warmup_steps=100,
    learning_rate=0.01,
    batch_size=None,
    ensemble_size=10,

    sindy_weight=0.1,
    sindy_alpha=0.0001,
    sindy_pruning_frequency=50,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.01,
    sindy_library_polynomial_degree=2,
    
    compiled_forward=False,
    verbose=True,
    save_path_spice=path_spice,
)


In [ ]:
# fitting SPICE with SINDy as RNN-regularizer
print(f"\nStarting training on {estimator.device}...")
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

Once a model is trained and saved we can load it from the pickle file using `SpiceEstimator.load_spice(path_to_model)`

In [ ]:
estimator.load_spice(path_spice)

The SPICE models can be printed to the console using `SpiceEstimator.print_spice_model(participant_id)` 

In [ ]:
# Print example SPICE model
participant_id = 0
print(f"\nExample SPICE model for participant {participant_id}:")
estimator.print_spice_model(participant_id=participant_id)

# Benchmark models

## Classic benchmark model 

In [ ]:
class benchmark(torch.nn.Module):
    
    def __init__(
        self,
        n_actions,
        n_participants,
        **kwargs,
        ):
        super().__init__()
    
    def forward(self, inputs, state=None):
        # inputs -> torch.Tensor of shape (sessions=n_participants*n_blocks, trials, timesteps_in_trial=1, features)
        
        logits, updated_state = None, None
        
        # something happens here
        return logits, updated_state

## GRU benchmark model

In [ ]:
gru = GRU(n_actions=n_actions, additional_inputs=2, dropout=0.25)#.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
path_gru = '../params/ganesh2024a/gru_ganesh2024a.pkl'

In [ ]:
epochs = 1000
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    gru=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    device=torch.device('cpu'),
    )

torch.save(gru.state_dict(), path_gru)
print("Trained GRU parameters saved to " + path_gru)

In [ ]:
state_dict = torch.load(path_gru)
gru.load_state_dict(state_dict)

# Analysis

In [ ]:
import sys

sys.path.append("../..")
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation

## Analysis of trial averaged accuracy

In [ ]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    gru_model=gru,
    # benchmark_model=benchmark,
    )

## Analysis of individual differences (only SPICE)

This function analyses individual differences of the SPICE models w.r.t. the fitted SINDy coefficients.

Here you can choose between a discrete odds ratio analysis between groups (e.g. `HighAccumulatedReward` vs `LowAccumulatedReward`) or a beta-effect analysis of a continuous variable (e.g. `AccumulatedReward`).

The analyses check whether the given criterion is a good predictor of the SINDy coefficients in the model.

In [ ]:
analysis_coefficients_individuals(
    path_model=path_spice,
    path_data=path_data,
    criterion="SomePerformanceColumnInDataset",
    analysis="disc", # also: "cont"
    reference="ReferenceGroupFromPerformanceColumn", # only necessary if analysis="disc"
    dir_output="../data/ganesh2024a/",
    model_class=SPICERNN,
    model_config=spice_config,
)

## Plot SPICE against benchmark models

In [ ]:
# plotting
participant_id = 7

estimator.print_spice_model(participant_id)

agents = {
    # add baseline agent here
    'rnn': estimator.rnn_agent,
    'spice': estimator.spice_agent,
    'gru': gru_agent,
}

fig, axs = plot_session(agents, dataset.xs[participant_id])
plt.show()